# CIFAR-10N Uncertainty Visualization - 3 Classes

**Simplified version with 3 classes for clearer decision boundaries**

**Features:**
- Only 3 classes (airplane, automobile, bird) for clear visualization
- Decision boundaries visible in 2D
- Misclassified samples marked with X
- DINOv2 (768-dim) OR ResNet-10 (512-dim) feature extraction
- Aleatoric vs Epistemic uncertainty distinction

**The MLP is a fully connected network** operating on high-dimensional embeddings.

In [15]:
# ============================================================================
# CONFIGURATION - 3 Classes for Clear Visualization
# ============================================================================

FEATURE_EXTRACTOR = 'resnet10'  # 'dinov2' or 'resnet10'
SELECTED_CLASSES = [0, 1, 2]  # airplane, automobile, bird (3 classes only)
UNDER_SUPPORTED_CLASSES = [2]  # bird will be under-supported (epistemic)
UNDER_TRAIN_PER_CLASS = 10
REGULAR_TRAIN_PER_CLASS = 50
EPOCHS = 30  # More epochs for better convergence with fewer classes
MC_SAMPLES = 30
REDUCTION_METHOD = 'umap'  # 'tsne' or 'umap'
RANDOM_SEED = 42

print(f"✓ Config: {FEATURE_EXTRACTOR.upper()}, 3 classes, {EPOCHS} epochs")

✓ Config: RESNET10, 3 classes, 30 epochs


In [16]:
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.manifold import TSNE
try:
    import umap
except:
    print("⚠️ UMAP not available, will use t-SNE")

sys.path.insert(0, str(Path.cwd()))
from src.data.cifar10n_loader import CIFAR10NDataset
from uq_classification.data_loader import sample_indices_for_fast_pilot
from uq_classification.models import EmbeddingDataset, EmbeddingDropoutMLP

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

⚠️ UMAP not available, will use t-SNE
Device: cpu


In [17]:
# Load CIFAR-10N (reuses downloaded data)
from torchvision import transforms

# Define transforms
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

dataset = CIFAR10NDataset(
    root='./backend/data/cifar10n', 
    train=True, 
    noise_type='worse_label',
    transform=transform,
    download=True
)

# Filter to only 3 classes
all_labels = np.array(dataset.cifar10.targets)
mask_3class = np.isin(all_labels, SELECTED_CLASSES)
indices_3class = np.where(mask_3class)[0]

# Create filtered dataset
class FilteredDataset:
    def __init__(self, dataset, indices, class_mapping):
        self.dataset = dataset
        self.indices = indices
        self.class_mapping = class_mapping  # Map original labels to 0,1,2
        # Compute noise mask for filtered indices
        # Create a fake cifar10 object with remapped targets for compatibility
        class FakeCIFAR10:
            def __init__(self, original_targets, indices, class_mapping):
                # Get targets for filtered indices and remap them
                filtered_targets = np.array(original_targets)[indices]
                self.targets = [class_mapping[t] for t in filtered_targets]
        self.cifar10 = FakeCIFAR10(dataset.cifar10.targets, indices, class_mapping)
        
        self.noise_mask = dataset.noise_mask[indices]
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        img, label, clean, noisy = self.dataset[self.indices[idx]]
        # Remap labels to 0,1,2 (handle both int and tensor)
        label_val = label.item() if isinstance(label, torch.Tensor) else label
        clean_val = clean.item() if isinstance(clean, torch.Tensor) else clean
        label = self.class_mapping[label_val]
        clean = self.class_mapping[clean_val]
        return img, torch.tensor(label), torch.tensor(clean), noisy

class_mapping = {0: 0, 1: 1, 2: 2}  # airplane->0, automobile->1, bird->2
dataset = FilteredDataset(dataset, indices_3class, class_mapping)

class_names = ['airplane', 'automobile', 'bird']
print(f"Dataset: {len(dataset)} samples (3 classes only), {dataset.noise_mask.sum()} noisy")

Loaded CIFAR-10N with worse_label: 40.21% noise
Dataset: 15000 samples (3 classes only), 5628 noisy


In [18]:
# Sample train/eval splits
split_spec = sample_indices_for_fast_pilot(
    dataset, under_supported_classes=UNDER_SUPPORTED_CLASSES,
    under_train_per_class=UNDER_TRAIN_PER_CLASS,
    regular_train_per_class=REGULAR_TRAIN_PER_CLASS,
    eval_per_group=50, seed=RANDOM_SEED
)
print(f"Train: {len(split_spec.train_indices)}, Eval: {len(split_spec.clean_eval_indices)+len(split_spec.aleatoric_eval_indices)+len(split_spec.epistemic_eval_indices)}")

Train: 110, Eval: 150


In [19]:
# Feature extractor
class ResNet10Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(pretrained=True)
        self.features = nn.Sequential(*list(resnet.children())[:-1])
        self.feature_dim = 512
    def forward(self, x):
        return self.features(x).squeeze(-1).squeeze(-1)

def extract_features(dataset, indices, backbone_type, batch_size=32):
    subset = Subset(dataset, list(indices))
    loader = DataLoader(subset, batch_size=batch_size, shuffle=False)
    
    if backbone_type == 'dinov2':
        from src.models.dinov2_backbone import create_dinov2_model
        model = create_dinov2_model('dinov2_vits14', 3, 0.0, False, True).to(device)  # 3 classes!
        extract_fn = lambda imgs: model.extract_features(imgs.to(device))
        feat_dim = 768
    else:  # resnet10
        model = ResNet10Extractor().to(device)
        extract_fn = lambda imgs: model(imgs.to(device))
        feat_dim = 512
    
    model.eval()
    all_feats, all_labels, all_clean, all_noisy = [], [], [], []
    
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"Extracting {backbone_type}"):
            imgs, labels, clean, noisy = batch
            all_feats.append(extract_fn(imgs).cpu())
            all_labels.append(labels.cpu())
            all_clean.append(clean.cpu())
            all_noisy.append(noisy.cpu())
    
    return {
        'features': torch.cat(all_feats),
        'labels': torch.cat(all_labels),
        'clean_labels': torch.cat(all_clean),
        'is_noisy': torch.cat(all_noisy).bool()
    }

# Extract for all indices
all_idx = np.concatenate([split_spec.train_indices, split_spec.clean_eval_indices, 
                          split_spec.aleatoric_eval_indices, split_spec.epistemic_eval_indices])
feat_dict = extract_features(dataset, all_idx, FEATURE_EXTRACTOR)

n_train = len(split_spec.train_indices)
train_feats = feat_dict['features'][:n_train]
eval_feats = feat_dict['features'][n_train:]
feature_dim = train_feats.shape[1]
print(f"✓ Features: {feature_dim}-dim, train={train_feats.shape}, eval={eval_feats.shape}")

/Users/andrearachetta/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/Users/andrearachetta/Library/Python/3.9/lib/python/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Extracting resnet10:   0%|          | 0/9 [00:00<?, ?it/s]


AttributeError: 'int' object has no attribute 'item'

In [ ]:
# Train MLP with MC Dropout (3 classes!)
train_ds = EmbeddingDataset(train_feats, feat_dict['labels'][:n_train])
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

model = EmbeddingDropoutMLP(feature_dim, 3, dropout_p=0.3).to(device)  # 3 classes!
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for feats, labels in train_loader:
        feats, labels = feats.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(feats)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

print("✓ Training complete!")

In [ ]:
# MC Dropout inference
model.train()  # Keep dropout active
mc_probs = []
with torch.no_grad():
    for _ in tqdm(range(MC_SAMPLES), desc="MC Dropout"):
        logits = model(eval_feats.to(device))
        probs = torch.softmax(logits, 1)
        mc_probs.append(probs.cpu())

mc_probs = torch.stack(mc_probs)  # [MC_SAMPLES, N, 3]
mean_probs = mc_probs.mean(0)
pred_classes = mean_probs.argmax(1)
pred_ent = -(mean_probs * torch.log(mean_probs + 1e-10)).sum(1)

print(f"✓ MC Dropout complete: {mc_probs.shape}")

In [ ]:
# Dimensionality reduction
all_feats_np = feat_dict['features'].numpy()

if REDUCTION_METHOD == 'umap' and 'umap' in sys.modules:
    reducer = umap.UMAP(n_components=2, random_state=RANDOM_SEED)
    reduced = reducer.fit_transform(all_feats_np)
    print(f"✓ Reduced to 2D using {REDUCTION_METHOD}")
else:
    if REDUCTION_METHOD == 'umap':
        print("⚠️ UMAP failed, using t-SNE")
    reducer = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30)
    reduced = reducer.fit_transform(all_feats_np)
    print(f"✓ Reduced to 2D using t-SNE")

eval_reduced = reduced[n_train:]

In [ ]:
# Visualize with misclassification markers
n_clean = len(split_spec.clean_eval_indices)
n_aleat = len(split_spec.aleatoric_eval_indices)
eval_type = torch.zeros(len(eval_feats), dtype=torch.long)
eval_type[:n_clean] = 0  # Clean
eval_type[n_clean:n_clean+n_aleat] = 1  # Aleatoric
eval_type[n_clean+n_aleat:] = 2  # Epistemic

# Check which samples are misclassified
misclassified = (pred_classes != feat_dict['clean_labels'][n_train:]).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['green', 'orange', 'red']
labels = ['Clean', 'Aleatoric', 'Epistemic']

# Plot 1: By uncertainty type with X for misclassified
for i, (c, l) in enumerate(zip(colors, labels)):
    mask = eval_type == i
    # Correctly classified
    correct_mask = mask & ~misclassified
    axes[0].scatter(eval_reduced[correct_mask, 0], eval_reduced[correct_mask, 1], 
                    c=c, label=l, alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    # Misclassified with X marker
    wrong_mask = mask & misclassified
    if wrong_mask.sum() > 0:
        axes[0].scatter(eval_reduced[wrong_mask, 0], eval_reduced[wrong_mask, 1], 
                        c=c, alpha=0.8, s=100, marker='x', linewidths=3)

axes[0].set_title(f'Uncertainty Types (X = misclassified)', fontsize=14, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: By true class with X for misclassified
for cls in range(3):
    mask = feat_dict['clean_labels'][n_train:].numpy() == cls
    correct_mask = mask & ~misclassified
    axes[1].scatter(eval_reduced[correct_mask, 0], eval_reduced[correct_mask, 1],
                    c=f'C{cls}', label=class_names[cls], alpha=0.6, s=50, edgecolors='black', linewidth=0.5)
    wrong_mask = mask & misclassified
    if wrong_mask.sum() > 0:
        axes[1].scatter(eval_reduced[wrong_mask, 0], eval_reduced[wrong_mask, 1],
                        c=f'C{cls}', alpha=0.8, s=100, marker='x', linewidths=3)

axes[1].set_title('True Classes (X = misclassified)', fontsize=14, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
axes[1].grid(True, alpha=0.3)

# Plot 3: Predictive entropy
scatter = axes[2].scatter(eval_reduced[:, 0], eval_reduced[:, 1], 
                          c=pred_ent.cpu().numpy(), cmap='YlOrRd', alpha=0.6, s=50)
# Mark misclassified with X
if misclassified.sum() > 0:
    axes[2].scatter(eval_reduced[misclassified, 0], eval_reduced[misclassified, 1],
                    c='black', alpha=0.8, s=100, marker='x', linewidths=3, label='Misclassified')
axes[2].set_title('Predictive Entropy', fontsize=14, fontweight='bold')
plt.colorbar(scatter, ax=axes[2])
axes[2].legend(loc='best', fontsize=10)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"✓ Misclassification rate: {misclassified.sum()}/{len(misclassified)} ({misclassified.mean()*100:.1f}%)")

In [ ]:
# Decision boundaries with misclassification markers
from scipy.spatial import cKDTree

x_min, x_max = reduced[:, 0].min() - 1, reduced[:, 0].max() + 1
y_min, y_max = reduced[:, 1].min() - 1, reduced[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 100), np.linspace(y_min, y_max, 100))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

tree = cKDTree(reduced)
_, indices = tree.query(grid_pts, k=1)
grid_feats = torch.from_numpy(all_feats_np[indices]).float()

with torch.no_grad():
    model.eval()
    grid_logits = model(grid_feats.to(device))
    grid_probs = torch.softmax(grid_logits, 1)
    grid_preds = grid_probs.argmax(1).cpu().numpy()
    grid_conf = grid_probs.max(1)[0].cpu().numpy()

Z_class = grid_preds.reshape(xx.shape)
Z_conf = grid_conf.reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Plot 1: Decision boundaries by class
axes[0].contourf(xx, yy, Z_class, levels=np.arange(4)-0.5, cmap='tab10', alpha=0.3)
# Plot correctly classified
for cls in range(3):
    mask = (feat_dict['clean_labels'][n_train:].numpy() == cls) & ~misclassified
    axes[0].scatter(eval_reduced[mask, 0], eval_reduced[mask, 1], 
                    c=f'C{cls}', s=100, edgecolors='black', linewidth=1.5, alpha=0.8, label=class_names[cls])
# Plot misclassified with X
if misclassified.sum() > 0:
    axes[0].scatter(eval_reduced[misclassified, 0], eval_reduced[misclassified, 1],
                    c='red', s=150, marker='x', linewidths=3, label='Misclassified')
axes[0].set_title(f'Decision Boundaries - 3 Classes ({FEATURE_EXTRACTOR.upper()})', fontsize=14, fontweight='bold')
axes[0].legend(loc='best', fontsize=10)
axes[0].grid(True, alpha=0.3)

# Plot 2: Confidence map
axes[1].contourf(xx, yy, Z_conf, levels=20, cmap='RdYlGn', alpha=0.6)
for i, (c, l) in enumerate(zip(colors, labels)):
    mask = (eval_type == i) & ~misclassified
    axes[1].scatter(eval_reduced[mask, 0], eval_reduced[mask, 1], 
                    c=c, label=l, s=100, edgecolors='black', linewidth=1.5, alpha=0.8)
# Mark misclassified
if misclassified.sum() > 0:
    axes[1].scatter(eval_reduced[misclassified, 0], eval_reduced[misclassified, 1],
                    c='red', s=150, marker='x', linewidths=3, label='Misclassified')
axes[1].set_title('Confidence Map', fontsize=14, fontweight='bold')
axes[1].legend(loc='best', fontsize=10)
cbar = plt.colorbar(axes[1].collections[0], ax=axes[1])
cbar.set_label('Confidence', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("✓ Visualization complete!")

## Summary

This simplified 3-class notebook demonstrated:
1. **Feature extraction** with DINOv2 (768-dim) or ResNet-10 (512-dim)
2. **3-class classification** (airplane, automobile, bird) for clear boundaries
3. **Misclassification visualization** with X markers
4. **MC Dropout** for epistemic/aleatoric decomposition
5. **2D decision boundaries** via t-SNE/UMAP

### Key Points:
- Only 3 classes makes decision boundaries much clearer
- X markers show misclassified samples
- The MLP is a **fully connected network** (not convolutional)
- Decision boundaries are **non-linear** due to ReLU activation
- Change `FEATURE_EXTRACTOR` to compare DINOv2 vs ResNet-10